In [4]:
"""
Usage example for the polydiff PolyTensor MVP.

Run:
  PYTHONPATH=. python examples/usage_example.py

This example stays within the MVP op set: elementwise + and * (and backward).
No reductions like sum/mean are used because they are not implemented here.
"""

import torch

from polytensor import PolyTensor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

deg = 2
lr = 0.1

In [ ]:
# Parameter: w(z) = 2.0 + 0.01 z  (degree 2; higher coefficients 0)
w = PolyTensor.from_primal(
    torch.tensor(2.0, device=device),
    degree=deg,
    tangent=torch.tensor(0.01, device=device),
    requires_grad=True,
)

# Plain Tensor constant; will be promoted to a constant polynomial internally.
x = torch.tensor(3.0, device=device)

In [ ]:
# Arithmetic-only scalar "loss":
# loss = (w*x + w) * w = (x+1) * w^2
loss = (w * x + w) * w

print("device:", device)
print("w coeffs:", w.coeffs)
print("loss coeffs:", loss.coeffs)

loss.backward()

print("w.grad coeffs:", w.grad.coeffs)  # type: ignore[union-attr]

# Expected (for intuition):
# loss = 4*w^2  (since x=3)
# dloss/dw = 8*w, so at w(z)=2+0.01z:
# grad(z) = 16 + 0.08 z  (degree-2 coefficient stays 0)

# Optimizer-like step.
with torch.no_grad():
    w.add_(w.grad, alpha=-lr)  # w -= lr * grad  (updates all polynomial coefficients)
    w.grad = None

print("w after step coeffs:", w.coeffs)

device: cpu
w coeffs: tensor([2.0000, 0.0100, 0.0000])
loss coeffs: tensor([1.6000e+01, 1.6000e-01, 4.0000e-04])
w.grad coeffs: tensor([16.0000,  0.0800,  0.0000])
w after step coeffs: tensor([0.4000, 0.0020, 0.0000])
